# core

> Feature-complete ECharts components for FastHTML — with JS formatters, HTMX click events, theme support, and memory cleanup.

In [ ]:
#| default_exp core

In [ ]:
#| export
import json
import uuid

from fasthtml.common import Script, Div, NotStr, to_xml

In [ ]:
#| hide
from nbdev.showdoc import *

## Header

In [ ]:
#| export
DEFAULT_ECHARTS_VERSION = "5.5.0"

def echarts_header(version=DEFAULT_ECHARTS_VERSION):
    "Include the ECharts CDN script tag in your FastHTML app headers."
    return Script(src=f"https://cdn.jsdelivr.net/npm/echarts@{version}/dist/echarts.min.js")

Include `echarts_header()` in your FastHTML app's `hdrs` to load the ECharts library:

```python
app, rt = fast_app(hdrs=(echarts_header(),))
```

## JavaScript Function Support

ECharts relies heavily on JavaScript functions for custom tooltips, axis labels, etc.
Since `json.dumps` turns everything into strings, we need a way to mark certain strings
as raw JavaScript. The `JSFunc` wrapper and `EChartsEncoder` handle this.

In [ ]:
#| export
class JSFunc(str):
    "Marker class to flag a string as a raw JavaScript function for ECharts options."

In [ ]:
#| export
class EChartsEncoder(json.JSONEncoder):
    "Custom JSON encoder that flags `JSFunc` values with a `!JS!` prefix so the frontend can revive them."
    def encode(self, o):
        return super().encode(self._convert(o))

    def _convert(self, o):
        if isinstance(o, JSFunc): return f"!JS!{str(o)}"
        if isinstance(o, dict): return {k: self._convert(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return [self._convert(i) for i in o]
        return o


For example, you can use `JSFunc` to pass a custom tooltip formatter:

In [ ]:
opts = {"tooltip": {"formatter": JSFunc("function(p) { return p.name + ': $' + p.value; }")}}
encoded = json.dumps(opts, cls=EChartsEncoder)
assert '!JS!' in encoded
print(encoded)

{"tooltip": {"formatter": "!JS!function(p) { return p.name + ': $' + p.value; }"}}


## EChart Component

In [ ]:
#| export
def EChart(options: dict, chart_id: str = None, width: str = "100%", height: str = "400px",
           theme: str = None, hx_get_click: str = None, hx_target_click: str = None,
           hx_click_vals: list = None, hx_click_cb: str = None):
    "Render an EChart with support for themes, JS formatters, HTMX click events, and memory cleanup."
    chart_id = chart_id or f"echart_{uuid.uuid4().hex}"

    # Use our custom encoder to handle JSFunc objects safely
    safe_options = json.dumps(options, cls=EChartsEncoder)

    # Optional theme logic (e.g., 'dark' or 'light')
    theme_str = f"'{theme}'" if theme else "null"

    # Optional HTMX click handler logic
    click_logic = ""
    if hx_get_click:
        target = f"target: '{hx_target_click}', " if hx_target_click else ""
        if hx_click_cb:
            vals_js = f"var vals = ({hx_click_cb})(params);"
        else:
            fields = hx_click_vals or ["name", "value", "seriesName"]
            fields_js = json.dumps(fields)
            vals_js = f"var vals = {{}}; {fields_js}.forEach(function(k) {{ vals[k] = params[k]; }});"
        click_logic = f"""
        myChart.on('click', function(params) {{
            {vals_js}
            htmx.ajax('GET', '{hx_get_click}', {{
                {target}
                values: vals
            }});
        }});
        """

    js_code = f"""
    (function() {{
        const dom = document.getElementById('{chart_id}');
        if (!dom) return;

        const myChart = echarts.init(dom, {theme_str});

        const option = {safe_options};
        function reviveJS(obj) {{
            for (let k in obj) {{
                if (typeof obj[k] === 'object' && obj[k] !== null) reviveJS(obj[k]);
                else if (typeof obj[k] === 'string' && obj[k].startsWith('!JS!')) {{
                    try {{ obj[k] = eval('(' + obj[k].slice(4) + ')'); }} catch(e) {{ console.error(e); }}
                }}
            }}
        }}
        reviveJS(option);

        myChart.setOption(option);

        {click_logic}

        const resizeObserver = new ResizeObserver(() => {{ myChart.resize(); }});
        resizeObserver.observe(dom);

        dom.addEventListener('htmx:beforeCleanupElement', function() {{
            resizeObserver.disconnect();
            myChart.dispose();
        }});
    }})();
    """

    return Div(
        Div(id=chart_id, style=f"width: {width}; height: {height};"),
        Script(NotStr(js_code))
    )

In [ ]:
#| export
def preview_echart(echart, height="450px"):
    from IPython.display import HTML

    raw_html = to_xml(echart)
    srcdoc = f'<script src="https://cdn.jsdelivr.net/npm/echarts@{DEFAULT_ECHARTS_VERSION}/dist/echarts.min.js"></script>{raw_html}'
    srcdoc = srcdoc.replace("'", "&#39;")
    return HTML(f"<iframe srcdoc='{srcdoc}' style='width:100%;height:{height};border:none;'></iframe>")

A basic bar chart:

In [ ]:
options = {
    "xAxis": {"type": "category", "data": ["Mon", "Tue", "Wed", "Thu", "Fri"]},
    "yAxis": {"type": "value"},
    "series": [{"data": [120, 200, 150, 80, 70], "type": "bar"}]
}
chart = EChart(options, chart_id="demo_bar")
html = to_xml(chart)
assert 'id="demo_bar"' in html
assert 'echarts.init' in html

preview_echart(chart)

/usr/local/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


With dark theme and a JS tooltip formatter:

In [ ]:
options_dark = {
    "title": {"text": "Sales"},
    "tooltip": {
        "formatter": JSFunc("function(p) { return '<b>' + p.name + '</b>: $' + p.value; }")
    },
    "xAxis": {"data": ["Shirts", "Sweaters", "Hats"]},
    "yAxis": {},
    "series": [{"type": "bar", "data": [5, 20, 36]}]
}
chart_dark = EChart(options_dark, chart_id="demo_dark", theme="dark")
html_dark = to_xml(chart_dark)
assert "'dark'" in html_dark
assert '!JS!' in html_dark
print('Theme and JSFunc working ✓')

preview_echart(chart_dark)

Theme and JSFunc working ✓


With HTMX click integration:

In [ ]:
chart_htmx = EChart(options, chart_id="demo_htmx",
                     hx_get_click="/bar-clicked", hx_target_click="#result")
html_htmx = to_xml(chart_htmx)
assert '/bar-clicked' in html_htmx
assert "#result" in html_htmx
print('HTMX click integration working ✓')

HTMX click integration working ✓


With custom click fields (e.g. for multiple time series):

In [ ]:
chart_fields = EChart(options, chart_id="demo_fields",
                      hx_get_click="/clicked",
                      hx_click_vals=["name", "seriesIndex", "dataIndex", "data"])
html_fields = to_xml(chart_fields)
assert 'seriesIndex' in html_fields
assert 'dataIndex' in html_fields
print('Custom click fields working ✓')

Custom click fields working ✓


With a custom JS click callback for full control:

In [ ]:
chart_cb = EChart(options, chart_id="demo_cb",
                  hx_get_click="/clicked",
                  hx_click_cb=JSFunc("function(params) { return {x: params.data[0], y: params.data[1], series: params.seriesName}; }"))
html_cb = to_xml(chart_cb)
assert 'params.data[0]' in html_cb
assert 'params.seriesName' in html_cb
print('Custom click callback working ✓')

Custom click callback working ✓


## EChartUpdate

In [ ]:
#| export
def EChartUpdate(chart_id: str, options: dict, merge: bool = True):
    "Update an existing chart instance. Set `merge=False` to replace all options instead of merging."
    safe_options = json.dumps(options, cls=EChartsEncoder)
    not_merge_str = "false" if merge else "true"

    js_code = f"""
    (function() {{
        const dom = document.getElementById('{chart_id}');
        if (!dom) return;
        const myChart = echarts.getInstanceByDom(dom);
        if (myChart) {{
            const option = {safe_options};
            function reviveJS(obj) {{
                for (let k in obj) {{
                    if (typeof obj[k] === 'object' && obj[k] !== null) reviveJS(obj[k]);
                    else if (typeof obj[k] === 'string' && obj[k].startsWith('!JS!')) {{
                        try {{ obj[k] = eval('(' + obj[k].slice(4) + ')'); }} catch(e) {{}}
                    }}
                }}
            }}
            reviveJS(option);
            myChart.setOption(option, {not_merge_str});
        }}
    }})();
    """
    return Script(NotStr(js_code))

In [ ]:
update = EChartUpdate("demo_bar", {"series": [{"data": [300, 400, 500, 600, 700]}]})
html_update = to_xml(update)
assert 'getInstanceByDom' in html_update
assert '300' in html_update

## EChartJS

In [ ]:
#| export
def EChartJS(chart_id: str, js_func: str):
    "Run a JS function against an existing chart instance. `js_func` receives `(chart, el)` as arguments."
    js_code = f"""
    (function() {{
        const el = document.getElementById('{chart_id}');
        if (!el) return;
        const chart = echarts.getInstanceByDom(el);
        if (!chart) return;
        ({js_func})(chart, el);
    }})();
    """
    return Script(NotStr(js_code))

In [ ]:
# Stash data on the DOM element
js_stash = EChartJS("demo_bar", "function(chart, el) { el._loadData = chart.getOption().series[0].data; }")
html_stash = to_xml(js_stash)
assert '_loadData' in html_stash
assert 'getInstanceByDom' in html_stash

# Apply CSS filter
js_blur = EChartJS("demo_bar", "function(chart, el) { el.style.filter = 'blur(4px)'; }")
html_blur = to_xml(js_blur)
assert 'blur' in html_blur
print('EChartJS working ✓')

EChartJS working ✓


## EChartOOB

In [ ]:
#| export
def EChartOOB(*scripts, sink_id: str = "script-sink"):
    "Wrap one or more EChart Script elements in an OOB-swappable Div for HTMX responses."
    return Div(*scripts, id=sink_id, hx_swap_oob="true")

In [ ]:
oob = EChartOOB(
    EChartUpdate("demo_bar", {"series": [{"data": [100]}]}),
    EChartJS("demo_bar", "function(chart, el) { el.style.filter = ''; }")
)
html_oob = to_xml(oob)
assert 'hx-swap-oob="true"' in html_oob
assert 'script-sink' in html_oob
assert html_oob.count('<script>') == 2
print('EChartOOB working ✓')

EChartOOB working ✓


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()